# Spark SQL Queries by Table

This notebook reads table configurations from `config.json` and executes a `SELECT *` query on each table 
with the filter condition specified in the config and a `LIMIT 20`. The results are displayed in separate notebook cells.

**Configuration**: The notebook reads from `app/aws-glue/src/dependencies/config/config.json` which contains:
- Table source names
- Filter conditions (in the `filter` field)
- Target catalog/database/table information

**Note**: This notebook assumes an AWS Glue interactive session with Spark already configured.

In [ ]:
%glue_version 5.0
%iam_role arn:aws:iam::331504768406:role/role-glue-job-flight-radar
%region us-east-1
%worker_type G.1X
%number_of_workers 2
%idle_timeout 10
%session_id_prefix spark-sql-queries

In [ ]:
%%configure
{
  "--datalake-formats": "delta",
  "--conf": "spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension --conf spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog --conf spark.delta.logStore.class=org.apache.spark.sql.delta.storage.S3SingleDriverLogStore"
}

In [ ]:
%%tags
{"Environment": "production", "Project": "flight-radar-glue", "Mode": "batch"}

In [ ]:
%extra_py_files s3://lakehouse-workspace-331504768406/aws-glue/jobs/flight-radar/src/dependencies/helpers.zip

In [ ]:
from pyspark.context import SparkContext
from awsglue.context import GlueContext

sc = SparkContext.getOrCreate()
glue_context = GlueContext(sc)
spark = glue_context.spark_session

In [1]:
# Read the table configuration from S3 using the current AWS account
import boto3
from src.dependencies.config_models import Config

account_id = boto3.client("sts").get_caller_identity()["Account"]
config = Config.from_s3(
    f"s3://lakehouse-workspace-{account_id}/aws-glue/jobs/flight-radar/src/dependencies/config/config.json"
)

for source in config.sources:
    print(
        f"{source.order:>2}  {source.source:<20} -> "
        f"{source.target.database}.{source.target.table} "
        f"(filter: '{source.filter}')"
    )

Session spark-sql-queries-05427d90-fb93-4627-8263-3218167e9a75 has been created.
 1  aircraft             -> db_raw.fr_aircraft (filter: '')
 2  airports             -> db_raw.fr_airports (filter: '')
 3  airlines             -> db_raw.fr_airlines (filter: '')
 4  flights              -> db_raw.fr_flights (filter: '')
 5  aircraft_positions   -> db_raw.fr_aircraft_positions (filter: 'aircraft_icao24 >= "00" and aircraft_icao24 <= "99"')
 6  countries            -> db_raw.fr_countries (filter: '')
 7  aircraft_types       -> db_raw.fr_aircraft_types (filter: '')
 8  routes               -> db_raw.fr_routes (filter: '')


In [2]:
# Execute Spark SQL queries for each configured table with filter and limit 20
for source in config.sources:
    table_name = f"{source.target.database}.{source.target.table}"
    filter_condition = source.filter.strip() if source.filter else ""

    try:
        if not spark.catalog.tableExists(table_name):
            print(f"[SKIP] Table does not exist: {table_name}")
            continue

        if filter_condition:
            query = f"SELECT * FROM {table_name} WHERE {filter_condition} LIMIT 20"
        else:
            query = f"SELECT * FROM {table_name} LIMIT 20"

        print(f"Executing query for {source.source}: {query}")
        df = spark.sql(query)
        df.show(20, truncate=False)
    except Exception as exc:
        print(f"[ERROR] Failed to query {table_name}: {exc}")
        print(f"[SKIP] Continuing with the next table")

Executing query for aircraft: SELECT * FROM db_raw.fr_aircraft LIMIT 20
[ERROR] Failed to query db_raw.fr_aircraft: [DELTA_TABLE_NOT_FOUND] Delta table `db_raw`.`fr_aircraft` doesn't exist.
[SKIP] Continuing with the next table
Executing query for airports: SELECT * FROM db_raw.fr_airports LIMIT 20
[ERROR] Failed to query db_raw.fr_airports: [DELTA_TABLE_NOT_FOUND] Delta table `db_raw`.`fr_airports` doesn't exist.
[SKIP] Continuing with the next table
Executing query for airlines: SELECT * FROM db_raw.fr_airlines LIMIT 20
[ERROR] Failed to query db_raw.fr_airlines: [DELTA_TABLE_NOT_FOUND] Delta table `db_raw`.`fr_airlines` doesn't exist.
[SKIP] Continuing with the next table
Executing query for flights: SELECT * FROM db_raw.fr_flights LIMIT 20
[ERROR] Failed to query db_raw.fr_flights: [DELTA_TABLE_NOT_FOUND] Delta table `db_raw`.`fr_flights` doesn't exist.
[SKIP] Continuing with the next table
Executing query for aircraft_positions: SELECT * FROM db_raw.fr_aircraft_positions WHERE air

In [50]:
# Stop the session when done
%stop_session

Stopping session: spark-sql-queries-05427d90-fb93-4627-8263-3218167e9a75
Stopped session.
